In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

DATA_DIR = '../data'

customers = pd.read_csv(f'{DATA_DIR}/customers.csv')
accounts = pd.read_csv(f'{DATA_DIR}/accounts.csv')
transactions = pd.read_csv(f'{DATA_DIR}/transactions.csv', parse_dates=['timestamp'])
ground_truth = pd.read_csv(f'{DATA_DIR}/ground_truth.csv')
merchants = pd.read_csv(f'{DATA_DIR}/merchants.csv')
employers = pd.read_csv(f'{DATA_DIR}/employers.csv')
counterparties = pd.read_csv(f'{DATA_DIR}/counterparties.csv')

print(f"Customers: {customers.shape}")
print(f"Accounts: {accounts.shape}")
print(f"Transactions: {transactions.shape}")
print(f"Ground truth: {ground_truth.shape}")
print(f"Merchants: {merchants.shape}")
print(f"Employers: {employers.shape}")
print(f"Counterparties: {counterparties.shape}")

Customers: (10000, 13)
Accounts: (11937, 4)
Transactions: (5712869, 14)
Ground truth: (1576, 6)
Merchants: (500, 4)
Employers: (1000, 3)
Counterparties: (2000, 5)


In [4]:
customers.head()

,customer_id,segment,name,income_or_turnover,budget_pct,expected_monthly_outflow,expected_monthly_txns,salary_day,employer_id,fav_merchants,fav_counterparties,kyc_risk,join_date
0,CUST_100000,retail,Nathan Mccann,4846,0.825866,4002,9,26.0,EMP_1752,MERCH_1460|MERCH_1262|MERCH_1405|MERCH_1270|ME...,CP_10140|CP_11063,Low,2022-10-01
1,CUST_100001,retail,Julie Gonzalez,200000,0.709336,141867,20,26.0,EMP_1560,MERCH_1229|MERCH_1280|MERCH_1212|MERCH_1319,CP_10016|CP_10713,Medium,2019-02-01
2,CUST_100002,sme,Ortiz Inc,1738989,0.593891,1032770,128,NaN,NaN,MERCH_1044|MERCH_1003|MERCH_1002|MERCH_1215,CP_10301|CP_10059,Low,2020-06-01
3,CUST_100003,corporate,Brown Ltd,893317,0.853347,762309,103,NaN,NaN,MERCH_1172|MERCH_1461|MERCH_1475|MERCH_1034|ME...,CP_11047|CP_11658|CP_11472,Low,2021-12-01
4,CUST_100004,retail,Bobby Williams,17668,0.672978,11890,35,28.0,EMP_1126,MERCH_1057|MERCH_1152|MERCH_1025|MERCH_1478|ME...,CP_10685|CP_10300|CP_11783,Low,2022-10-01


customer_id - Core entity anchor for Customer Due Diligence (CDD) and historical profiling.

segment - Sets baseline behavioral expectations (turnover volume, velocity, transaction types).

name - Target for sanctions screening, adverse media checks, and Politically Exposed Person (PEP) lists.

income_or_turnover - Benchmarks transaction volume. Spikes beyond income trigger threshold/velocity alerts.

budget_pct - Identifies rapid pass-through or layering (e.g., 1.0 outflow immediately after deposit).

expected_monthly_outflow - Benchmark for unusual deviation in transaction amounts.

expected_monthly_txns - Detects sudden high-frequency activity (e.g., burst transactions, bot activity).

salary_day - Establishes periodic regularity; deviations indicate irregular cash injections.

employer_id - Validates declared source of funds/wealth against registered business entities.

fav_merchants - Establishes a baseline "normal" spending graph; uncharacteristic merchant deviations stand out.

fav_counterparties - Profiles habitual transfer networks; transactions with new or offshore entities raise risk scores.

kyc_risk - Determines monitoring thresholds and Enhanced Due Diligence (EDD) requirements.

join_date - Helps detect "dormant account activation" or immediate high-value activity on newly opened accounts.

In [5]:
accounts.head()

,account_id,customer_id,currency,status
0,ACC_1000000,CUST_100000,AED,Active
1,ACC_1000001,CUST_100001,AED,Active
2,ACC_1000002,CUST_100002,AED,Active
3,ACC_1000003,CUST_100003,USD,Active
4,ACC_1000004,CUST_100004,AED,Active


account_id - Ledger-level entity for tracking balance anomalies and transaction velocity.

customer_id - Links accounts to a single Ultimate Beneficial Owner (UBO) to detect multi-account structuring.

currency - Highlights foreign exchange layering, cross-border flows, and SWIFT reporting triggers.

status - Flags unusual activity spikes occurring on dormant or newly reactivated accounts.

In [7]:
transactions.head()

,transaction_id,account_id,customer_id,timestamp,transaction_type,channel,amount,currency,amount_aed_equivalent,direction,counterparty_id,counterparty_name,counterparty_country,balance_after
0,TXN_71C9B82D67,ACC_1000000,CUST_100000,2023-01-02 13:00:00,POS,Card,26.31,AED,26.31,Debit,MERCH_1248,Johnston-Hines Restaurant,United Arab Emirates,11041.59
1,TXN_8718134BDE,ACC_1000000,CUST_100000,2023-01-07 17:00:00,POS,Card,541.81,AED,541.81,Debit,MERCH_1248,Johnston-Hines Restaurant,United Arab Emirates,10499.78
2,TXN_78A2828EA9,ACC_1000000,CUST_100000,2023-01-12 18:00:00,Transfer,Online Banking,33.40,AED,33.40,Debit,CP_10140,Kim Morales,Egypt,10466.38
3,TXN_424AC624AF,ACC_1000000,CUST_100000,2023-01-14 10:00:00,POS,Card,201.56,AED,201.56,Debit,MERCH_1270,"Thomas, Lee and Greene Supermarket",United Arab Emirates,10264.82
4,TXN_2D36E06B7D,ACC_1000000,CUST_100000,2023-01-16 19:00:00,POS,Card,1033.05,AED,1033.05,Debit,MERCH_1247,"Matthews, Smith and Hubbard Fuel",United Arab Emirates,9231.77


transaction_id - Audit trail identifier for regulatory filings (SAR / STR submissions).

account_id - Isolates account-level balance velocity and threshold breaches.

customer_id - Enables cross-account aggregation for holistic entity-level monitoring.

timestamp - Drives time-window rules (e.g., 7-day rolling structuring, burst velocity).

transaction_type - Flags high-risk methods (cash deposits, overseas wires) vs. standard retail spends.

channel - Assigns channel delivery risk (e.g., cash desk vs. automated clearing vs. card).

amount - Core numerical feature for threshold rules and outlier detection models.

currency - Identifies multi-currency layering and conversion manipulation.

amount_aed_equivalent - Standardizes all amounts against local regulatory reporting limits (e.g., AED 50k cash rule).

direction - Measures inflow/outflow balance to calculate pass-through and rapid movement ratios.

counterparty_id - Maps fund flows across accounts to detect cyclic rings and smurfing hubs.

counterparty_name - Matches counterparties against sanction lists, shell companies, or high-risk entities.

counterparty_country - Flags geographic risk (FATF grey/blacklists, offshore havens, sanctioned nations).

balance_after - Exposes "wash" accounts and rapid drain-downs where balances drop to near-zero.

In [8]:
ground_truth.head()

,transaction_id,account_id,scenario_id,scenario_type,scenario_role,is_suspicious
0,TXN_D9C69271D1,ACC_1008457,SCEN_STR_5063ed,Structuring,structuring_deposit,True
1,TXN_2F7CA85B52,ACC_1008457,SCEN_STR_5063ed,Structuring,structuring_deposit,True
2,TXN_57EF2B1D13,ACC_1008457,SCEN_STR_5063ed,Structuring,structuring_deposit,True
3,TXN_288BDFE9AD,ACC_1008457,SCEN_STR_5063ed,Structuring,structuring_deposit,True
4,TXN_920B674DD8,ACC_1008457,SCEN_STR_5063ed,Structuring,structuring_deposit,True


transaction_id - Maps known evaluation labels directly to individual ledger entries.

account_id - Measures scenario detection coverage at the account level.

scenario_id - Bundles multi-leg transactions into an end-to-end illicit laundering case.


scenario_role - Identifies specific placement, layering, or integration roles within a network.

is_suspicious - Target ground truth for training and benchmarking supervised AML models.

In [9]:
merchants.head()

,merchant_id,merchant_name,category,country
0,MERCH_1000,"Rodriguez, Figueroa and Sanchez Utility",Utility,United Arab Emirates
1,MERCH_1001,Doyle Ltd Luxury,Luxury,United Arab Emirates
2,MERCH_1002,"Mcclain, Miller and Henderson Fuel",Fuel,United Arab Emirates
3,MERCH_1003,Davis and Sons Restaurant,Restaurant,United Arab Emirates
4,MERCH_1004,"Guzman, Hoffman and Baldwin Supermarket",Supermarket,United Arab Emirates


merchant_id - Identifies the commercial recipient to differentiate retail spend from money transfers.

merchant_name - Used for entity resolution, merchant category verification, and adverse media screening.
    
category - Distinguishes normal daily spend (Supermarkets) from high-risk laundering vectors (Luxury, E-commerce).
    
country - Highlights cross-border card payments and foreign merchant routing risks.

In [10]:
employers.head()

,employer_id,employer_name,industry
0,EMP_1000,"Rivera, Martinez and Richardson LLC",Education
1,EMP_1001,Paul-Kline LLC,Tech
2,EMP_1002,Morgan-Lopez LLC,Tech
3,EMP_1003,"Jimenez, Glass and Stone LLC",Healthcare
4,EMP_1004,Acosta Inc LLC,Construction


employer_id - Verifies the declared payroll source during Source of Wealth / Funds verification.
    
employer_name - Screened against registered business registries and shell company indicators.
    
industry - Evaluates industry sector risk (e.g., cash-intensive construction/retail vs. corporate tech).

In [11]:
counterparties.head()

,cp_id,cp_name,cp_type,country,bank_name
0,CP_10000,Evans-Cannon Corp,Corporate,Germany,Harper-Robles Bank
1,CP_10001,"Brooks, Jenkins and Castro Holdings",Offshore,BVI,Wright-Espinoza Bank
2,CP_10002,"Price, Gray and Baker Corp",Corporate,Saudi Arabia,Garrett-Black Bank
3,CP_10003,"Aguirre, Cohen and Mitchell Trading",High-Risk,North Korea,"Patel, Adams and Lopez Bank"
4,CP_10004,Chad Rodriguez,Individual,UK,Williamson-Nelson Bank


cp_id - Graph node identifier for tracing cross-border and external beneficiary networks.
    
cp_name - Subject of watchlists, Politically Exposed Persons (PEP), and sanctions screening.
    
cp_type - Entity risk classification (Individual, Corporate, Offshore, High-Risk).
    
country - Primary indicator for high-risk jurisdictions, tax havens, and sanctioned states.
    
bank_name - Assesses correspondent banking risk and routing through high-risk financial institutions.

Now that I understand the relationships and schema of the data, lets dive into some of the common data quality issue. That is Orphan accounts, transaction, ground truth.
    Basically I'm asking myself, is there any accounts that doesn't have a customer id, any transaction that doesn't have account, any ground truth without matching transaction_id. The below query should return zero, if not it is a different data quality issue (similar to real life data) but for the scope of this projet, I'll remove all those data points as trying to fix that would be out of scope and will take up dispropotionate amount of time that cannot be justified considering what I'm aiming with this project

In [12]:
# accounts -> customers
orphan_accounts = ~accounts['customer_id'].isin(customers['customer_id'])
print(f"Accounts with no matching customer: {orphan_accounts.sum()}")

# transactions -> accounts
orphan_txns = ~transactions['account_id'].isin(accounts['account_id'])
print(f"Transactions with no matching account: {orphan_txns.sum()}")

# ground_truth -> transactions
orphan_gt = ~ground_truth['transaction_id'].isin(transactions['transaction_id'])
print(f"Ground truth rows with no matching transaction: {orphan_gt.sum()}")

Accounts with no matching customer: 0
Transactions with no matching account: 0
Ground truth rows with no matching transaction: 0


Seems like this generated data is not prone to that issue. Good. But to be frank I created the dataset so I matched everything with a forign key, so this wouldn't happen. 

There are some couter parties in transaction dataframe, which doesn't have any counterparties, that itself is not a redflag, especially these transactons could be bank fee, transaction within accounts, cash withdrawals. But it is important to highlight why this needs further investigation, the main reason is cash transaction, especially because cash breaks the digital trails, placement risks are especially high. If a electronic transfer/ wire doen't have counterparty details, that is a violation under CBUAE rules. 

In [13]:
print(f"Transactions with counterparty_id: {transactions['counterparty_id'].notna().mean():.1%}")

# Of those with a counterparty_id, how many resolve to merchants vs counterparties vs neither?
has_cp = transactions[transactions['counterparty_id'].notna()]
resolves_merchant = has_cp['counterparty_id'].isin(merchants['merchant_id'])
resolves_cp = has_cp['counterparty_id'].isin(counterparties['cp_id'])

print(f"Resolves to merchant: {resolves_merchant.mean():.1%}")
print(f"Resolves to counterparty: {resolves_cp.mean():.1%}")
print(f"Resolves to neither (e.g. internal account transfers): {(~resolves_merchant & ~resolves_cp).mean():.1%}")

Transactions with counterparty_id: 96.3%
Resolves to merchant: 78.3%
Resolves to counterparty: 19.6%
Resolves to neither (e.g. internal account transfers): 2.1%


By default the merchant transactions are low risk, most of the transaction in this category is day to day spends, low value high volume, but there will be transactions where the counterparty is operating in laxuary, casino, precious metal industry, which warrants more attention. Low values are always all the time genuine transactions

Counterparty transaction on the other hand posses high AML scrutiny. This is the core vector for layering and rapid movement of funds, evaluvated by geographical risk and peer comparison. 

    Internal transaction - without any reason - possible layering

    Null counterplay - High risk of placement and structuring. 

**Phase 2 - Risk Modeling Part 1 - Static Risk (Continues in the next notebook, more exciting things there)**